# 06주 · Hardy Cross·EPANET 관망

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: Hardy Cross 관망 해석 / EPANET·영상 병행 실습

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week06/CE272_W06.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week06) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 주제별 실습 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: 두 폐회로 Hardy Cross 반복표


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

<a id="s01"></a>

## 두 폐회로의 반복 계산

**먼저 예측**: 공유관에 서로 다른 두 유량을 저장해도 되는가?

**문제**: 원본 Excel의 관 5개, 초기 유량과 고정 마찰계수로 Hardy Cross 첫 보정량을 손계산하시오. 전역 유량 방향과 두 회로 방향을 표시한 뒤 반복표의 질량·수두잔차를 검토하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교


In [ ]:
trace = hyd.hardy_cross()
display(pd.DataFrame({'pipe':range(1,6),'L_m':hyd.HC_L,'D_m':hyd.HC_D,
                      'fixed_Darcy_f':hyd.HC_F,'initial_Q_m3_s':hyd.HC_Q0}))
display(trace)
print('Loop matrix:\n',hyd.HC_C)
print('Node balance A,B,C,D:',hyd.HC_B@trace.iloc[-1][['Q1','Q2','Q3','Q4','Q5']].to_numpy(dtype=float))

**검토 문제**: 공유관 보정에 두 회로의 보정량이 반대 부호로 들어가는 이유를 설명하시오. 보정량이 작다는 사실과 보존식 잔차가 작다는 사실을 구분하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분


<a id="s02"></a>

## EPANET 입력·단위·가정 대조

**먼저 예측**: EPANET 결과가 고정 f의 Excel과 다르면 프로그램 오류라고 결론 내릴 수 있는가?

**문제**: 주차 data 폴더의 EPANET 입력 파일과 경계조건을 확인하시오. EPANET에서 유량단위 LPS, 직경 mm, Darcy–Weisbach 조도 mm를 점검하고 고정 마찰계수 계산과의 가정 차이를 기록하시오. 모형의 조도·수두·유량 경계조건을 먼저 기록하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교


In [ ]:
table = pd.DataFrame({'pipe':range(1,6),'L_m':hyd.HC_L,'D_mm':hyd.HC_D*1000,
                      'epsilon_mm':[.045,.040,.040,.045,.042]})
Q = hyd.hardy_cross().iloc[-1][['Q1','Q2','Q3','Q4','Q5']].to_numpy(dtype=float)
table['Q_LPS'] = Q*1000
table['Re'] = [hyd.reynolds(q,d,1.006e-6) for q,d in zip(Q,hyd.HC_D)]
table['Colebrook_f_at_fixed_f_solution'] = [hyd.darcy_factor(re,eps/(d*1000)) for re,eps,d in zip(table['Re'],table['epsilon_mm'],hyd.HC_D)]
table['fixed_f_in_Excel'] = hyd.HC_F
table

**검토 문제**: 원본 zip의 실제 내용은 텍스트 .inp가 아니라 HC.net임을 확인하시오. 조도에서 재산정한 f와 원본 고정 f의 차이, 경계수두·유량단위의 차이를 분리하여 설명하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분


## 학습 기록

공유관 부호·절점 연속식·회로 수두잔차를 확인하고 모형 간 가정 차이를 설명하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록


In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')